In [ ]:
// ==== Setup ====

import java.util.*;

// wall-clock time of r, in milliseconds
long millis(Runnable r) {
    System.gc();            // start from a clean heap: less noise
    long start = System.nanoTime();
    r.run();
    return (System.nanoTime() - start) / 1_000_000;
}

# Stacks: recap

- **LIFO**: `push`, `pop`, `peek`, all at the **top**.
- Array-based (top at `size - 1`) or linked (top at the first node): `O(1)` per operation.

## Reading a stack

- The ADT restricts how a stack **changes**: only at the top.
- In practice, **read-only** iteration is usually allowed: from the top to the bottom, without changing the stack.

# Queues

## Queue ADT

- Elements are added at one end (**back**), removed at the other (**front**).
- **FIFO**: first in, first out.
- Examples: a line at a counter, jobs sent to a printer, requests to a server.

| operation | |
|---|---|
| `enqueue(e)` | add `e` at the back |
| `dequeue()` | remove and return the front element |
| `peek()` | return the front element, without removing it |
| `isEmpty()`, `size()` | |

In [ ]:
interface MyQueue<E> {
    void enqueue(E e);

    // throw NoSuchElementException if the queue is empty
    E dequeue();
    E peek();

    boolean isEmpty();
    int size();
}

## Linked queue

- Additions and removals at **different** ends.
- Singly linked, with `head` and `tail`:

| front at | `enqueue` | `dequeue` |
|---|---|---|
| first node | ? | ? |
| last node | ? | ? |

Front at the first node:
- `enqueue`: `add` at the tail
- `dequeue`: `removeFirst`
- Fields: `front`, `back`, `size`

In [ ]:
class LinkedQueue<E> implements MyQueue<E> {
    private static class Node<E> {
        E data;
        Node<E> next;       // the node behind this one

        Node(E data) { this.data = data; }
    }

    private Node<E> front;  // null if empty
    private Node<E> back;   // null if empty
    private int size;

    public int size() { return size; }
    public boolean isEmpty() { return front == null; }

    public void enqueue(E e) {
        Node<E> n = new Node<>(e);
        if (back == null) front = n;    // the queue was empty
        else back.next = n;
        back = n;
        size++;
    }

    public E dequeue() {
        if (front == null) throw new NoSuchElementException("empty queue");
        E e = front.data;
        front = front.next;
        if (front == null) back = null; // removed the only node
        size--;
        return e;
    }

    public E peek() {
        if (front == null) throw new NoSuchElementException("empty queue");
        return front.data;
    }

    // front to back
    @Override
    public String toString() {
        StringBuilder sb = new StringBuilder("[");
        for (Node<E> n = front; n != null; n = n.next) {
            sb.append(n.data);
            if (n.next != null) sb.append(", ");
        }
        return sb.append("]").toString();
    }
}

In [ ]:
MyQueue<String> q = new LinkedQueue<>();
q.enqueue("ann");
q.enqueue("bob");
q.enqueue("cat");
System.out.println(q + "  peek=" + q.peek());
System.out.println(q.dequeue() + " " + q.dequeue() + " " + q);

q.dequeue();                    // now empty
q.enqueue("dan");
System.out.println(q + "  size=" + q.size());

- Remove `if (front == null) back = null;` from `dequeue`: which sequence of calls breaks the queue?

## Array-based queue

### First attempt: front at index `0`
- `enqueue`: `data[size]`, `O(1)` amortized
- `dequeue`: `data[0]`, then shift everything left: `O(N)`

### Second attempt: a moving front
- `front`: index of the first element. `dequeue`: `front++`, no shift.
- Elements in `data[front..front+size-1]`.
- `enqueue` 8 and `dequeue` 6 times, capacity 8: what is wrong?

## Circular array

- After the last cell, continue from cell `0`.
- Element `i` (from the front): `data[(front + i) % capacity]`
- Next free cell: `data[(front + size) % capacity]`
- `dequeue`: `front = (front + 1) % capacity`
- Full only when `size == capacity`.

### Growing

- `Arrays.copyOf(data, 2 * capacity)` keeps the cells in the same positions. Why does it not work here?
- Copy element `i` from the front into cell `i`, then `front = 0`.

In [ ]:
class ArrayQueue<E> implements MyQueue<E> {
    private E[] data;
    private int front;      // index of the first element
    private int size;       // elements in data[front], data[front+1], ... wrapping around

    @SuppressWarnings("unchecked")
    ArrayQueue() { data = (E[]) new Object[4]; }

    public int size() { return size; }
    public boolean isEmpty() { return size == 0; }

    @SuppressWarnings("unchecked")
    private void grow() {
        E[] bigger = (E[]) new Object[2 * data.length];
        for (int i = 0; i < size; i++)
            bigger[i] = data[(front + i) % data.length];
        data = bigger;
        front = 0;
    }

    public void enqueue(E e) {
        if (size == data.length) grow();
        data[(front + size) % data.length] = e;
        size++;
    }

    public E dequeue() {
        if (size == 0) throw new NoSuchElementException("empty queue");
        E e = data[front];
        data[front] = null;
        front = (front + 1) % data.length;
        size--;
        return e;
    }

    public E peek() {
        if (size == 0) throw new NoSuchElementException("empty queue");
        return data[front];
    }

    // front to back
    @Override
    public String toString() {
        StringBuilder sb = new StringBuilder("[");
        for (int i = 0; i < size; i++) {
            sb.append(data[(front + i) % data.length]);
            if (i < size - 1) sb.append(", ");
        }
        return sb.append("]").toString();
    }

    // the array as it is, to see the wrap-around
    public String internals() {
        return Arrays.toString(data) + "  front=" + front + "  size=" + size;
    }
}

In [ ]:
ArrayQueue<Integer> aq = new ArrayQueue<>();
for (int i = 1; i <= 4; i++) aq.enqueue(i);
System.out.println(aq + "   " + aq.internals());

aq.dequeue();
aq.dequeue();
System.out.println(aq + "   " + aq.internals());

aq.enqueue(5);                  // wraps around
aq.enqueue(6);
System.out.println(aq + "   " + aq.internals());

aq.enqueue(7);                  // full: grows
System.out.println(aq + "   " + aq.internals());

- Cost of `enqueue`, `dequeue`, `peek`?
- Only `front` and `size` are fields. Where is the back?

## Array vs. linked queue

| | array (circular) | linked |
|---|---|---|
| `enqueue` | `O(1)` amortized, `O(N)` worst case | `O(1)` |
| `dequeue`, `peek` | `O(1)` | `O(1)` |
| memory | one cell per element, plus unused capacity | one node per element |

In [ ]:
System.out.printf("%10s %10s %10s%n", "N", "array ms", "linked ms");
for (int n = 1_000_000; n <= 8_000_000; n *= 2) {
    final int N = n;
    long arr = millis(() -> {
        MyQueue<Integer> q = new ArrayQueue<>();
        for (int i = 0; i < N; i++) q.enqueue(i % 10);    // small Integers are cached: no allocation
        while (!q.isEmpty()) q.dequeue();
    });
    long lnk = millis(() -> {
        MyQueue<Integer> q = new LinkedQueue<>();
        for (int i = 0; i < N; i++) q.enqueue(i % 10);
        while (!q.isEmpty()) q.dequeue();
    });
    System.out.printf("%,10d %10d %10d%n", N, arr, lnk);
}

- Compare with the stack timings: same picture?
- Which one would you choose? When would the other one be better?

# Deques

## Deque ADT

- **Double-ended queue**: add and remove at **both** ends.
- `addFirst`, `addLast`, `removeFirst`, `removeLast`, `peekFirst`, `peekLast`
- Stack: only one end. Queue: add at one end, remove at the other.

In [ ]:
interface MyDeque<E> {
    void addFirst(E e);
    void addLast(E e);

    // throw NoSuchElementException if the deque is empty
    E removeFirst();
    E removeLast();
    E peekFirst();
    E peekLast();

    boolean isEmpty();
    int size();
}

## Linked deque

- `MyDLinkedListDouble`: all six operations `O(1)` already.
- Singly linked, with `head` and `tail`: which operation is not `O(1)`?

## Circular array deque

`ArrayQueue` does `addLast` and `removeFirst`. The two new operations move the other way:
- `addFirst`: `front` moves one cell **back**
- `removeLast`: the last element is at `(front + size - 1) % capacity`

### `%` with negative numbers
- `front = 0`: `(front - 1) % capacity` is `-1`. In Java, `a % b` has the sign of `a`.
- `Math.floorMod(a, b)`: always in `0..b-1` for `b > 0`.

In [ ]:
System.out.println((-1 % 8) + " " + Math.floorMod(-1, 8));
System.out.println((-9 % 8) + " " + Math.floorMod(-9, 8));
System.out.println((13 % 8) + " " + Math.floorMod(13, 8));

In [ ]:
class MyArrayDeque<E> implements MyDeque<E> {
    private E[] data;
    private int front;
    private int size;

    @SuppressWarnings("unchecked")
    MyArrayDeque() { data = (E[]) new Object[4]; }

    public int size() { return size; }
    public boolean isEmpty() { return size == 0; }

    // index in data of element i, counting from the front
    private int index(int i) { return (front + i) % data.length; }

    @SuppressWarnings("unchecked")
    private void grow() {
        E[] bigger = (E[]) new Object[2 * data.length];
        for (int i = 0; i < size; i++)
            bigger[i] = data[index(i)];
        data = bigger;
        front = 0;
    }

    private void checkNotEmpty() {
        if (size == 0) throw new NoSuchElementException("empty deque");
    }

    public void addFirst(E e) {
        if (size == data.length) grow();
        front = Math.floorMod(front - 1, data.length);
        data[front] = e;
        size++;
    }

    public void addLast(E e) {
        if (size == data.length) grow();
        data[index(size)] = e;
        size++;
    }

    public E removeFirst() {
        checkNotEmpty();
        E e = data[front];
        data[front] = null;
        front = index(1);
        size--;
        return e;
    }

    public E removeLast() {
        checkNotEmpty();
        int last = index(size - 1);
        E e = data[last];
        data[last] = null;
        size--;
        return e;
    }

    public E peekFirst() { checkNotEmpty(); return data[front]; }
    public E peekLast()  { checkNotEmpty(); return data[index(size - 1)]; }

    @Override
    public String toString() {
        StringBuilder sb = new StringBuilder("[");
        for (int i = 0; i < size; i++) {
            sb.append(data[index(i)]);
            if (i < size - 1) sb.append(", ");
        }
        return sb.append("]").toString();
    }

    public String internals() {
        return Arrays.toString(data) + "  front=" + front + "  size=" + size;
    }
}

In [ ]:
MyArrayDeque<String> dq = new MyArrayDeque<>();
dq.addLast("c");
dq.addFirst("b");               // front wraps to the last cell
dq.addFirst("a");
System.out.println(dq + "   " + dq.internals());

dq.addLast("d");
dq.addLast("e");                // full: grows
System.out.println(dq + "   " + dq.internals());

System.out.println(dq.removeLast() + " " + dq.removeFirst() + " " + dq + "   " + dq.internals());

- `removeFirst` uses `index(1)`, `addFirst` uses `floorMod`. Why not `floorMod` in both?
- Cost of each operation?

## A list on a circular array

`MyArrayListDouble`, plus a `front` field: element `i` is in `data[(front + i) % capacity]`.

| | `MyArrayListDouble` | circular array |
|---|---|---|
| `get(i)`, `set(i, e)` | `O(1)` | `O(1)` |
| `add`, `remove` at the front | `O(N)` | `O(1)` amortized |
| `add`, `remove` at the back | `O(1)` amortized | `O(1)` amortized |
| `add`, `remove` in the middle | `O(N)` | `O(N)` |

- Both ends in `O(1)`, and still access by position in `O(1)`.
- In the middle: shift toward the **nearer** end, at most `N/2` moves.
- The price: one `+` and one `%` per access.

## Exercises

### 1. Tracing
Empty `ArrayQueue<String>`, capacity 4. Give `front`, `size` and the array after each operation.

```java
q.enqueue("A");
q.enqueue("B");
q.enqueue("C");
q.dequeue();
q.dequeue();
q.enqueue("D");
q.enqueue("E");
q.enqueue("F");
q.enqueue("G");     // what happens here?
q.dequeue();
```

### 2. Modular arithmetic
- By hand: `13 mod 8`, `-3 mod 8`, `-11 mod 8`, `24 mod 8`.
- In Java: `-3 % 8`, `-11 % 8`?
- Without `Math.floorMod`: an expression for `x mod n`, correct for `x >= -n`.
- `capacity = 10`, `front = 7`, `size = 5`: index of the next `enqueue`? Of the last element?

### 3. Costs

| | array stack | linked stack | array queue | linked queue |
|---|---|---|---|---|
| add | | | | |
| remove | | | | |
| peek | | | | |

Which entries differ between worst case and amortized cost?

In [ ]:
// test helpers
void check(boolean ok, String what) {
    System.out.println((ok ? "ok    " : "FAIL  ") + what);
}

### 4. Random access
Add `E get(int i)` to `ArrayQueue`: element `i` counting from the front, `0 <= i < size`. Cost? Same in `LinkedQueue`?

In [ ]:
ArrayQueue<Integer> g = new ArrayQueue<>();
for (int i = 0; i < 4; i++) g.enqueue(i);
g.dequeue();
g.dequeue();
g.enqueue(4);
g.enqueue(5);                   // wrapped: [2, 3, 4, 5]
check(g.get(0) == 2 && g.get(3) == 5, "after wrap: " + g + " " + g.internals());

### 5. Reversing a queue
`<E> void reverse(MyQueue<E> q)`: reverse the order of the elements in `q`, using one `MyArrayDeque<E>` as a stack: only `addFirst`, `removeFirst`, `isEmpty`.

In [ ]:
MyQueue<Integer> r = new ArrayQueue<>();
for (int i = 1; i <= 5; i++) r.enqueue(i);
reverse(r);
check(r.toString().equals("[5, 4, 3, 2, 1]"), "reversed: " + r);

### 6. A queue from two stacks
`TwoStackQueue<E> implements MyQueue<E>`, with two fields `MyArrayDeque<E> in, out` used as stacks: only `addFirst`, `removeFirst`, `peekFirst`, `isEmpty`, `size`.
- `enqueue`: push on `in`.
- `dequeue`: when is it necessary to move elements from `in` to `out`?
- Cost of a single `dequeue`, worst case? Total cost of `N` `enqueue` and `N` `dequeue`?

In [ ]:
MyQueue<Integer> tq = new TwoStackQueue<>();
tq.enqueue(1);
tq.enqueue(2);
tq.enqueue(3);
check(tq.dequeue() == 1, "first in, first out");
tq.enqueue(4);
check(tq.peek() == 2 && tq.size() == 3, "peek, size");
check(tq.dequeue() == 2 && tq.dequeue() == 3 && tq.dequeue() == 4, "mixed order");
check(tq.isEmpty(), "empty at the end");

### 7. Without `size`
Circular queue with fields `front` and `back` (next free cell), no `size`.
- `front == back`: empty or full?
- Fix: never fill the last cell. When is the queue full?
- `size()` from `front`, `back`, `capacity`?

### 8. A stack from two queues
`QueueStack<E>` with `push`, `pop`, `peek`, using two `MyQueue<E>` fields. Which operation is `O(N)`? Can both `push` and `pop` be `O(1)`?